# Sports Highlight Generator + Virtual Ad Insertion
## AMD AI Hackathon — TCS Track 2 (Multimodal)

**Complete end-to-end pipeline:** 3-modality highlight detection + LLM captions + Virtual Ad Insertion

| Component | Model | VRAM |
|-----------|-------|------|
| Speech recognition | Whisper large-v3 | ~10 GB |
| Frame scoring | EfficientNet-B0 | ~1 GB |
| Caption generation | Llama 3.1 8B (vLLM) | ~16 GB |
| Player occlusion | YOLOv8n-seg | ~1 GB |
| **Total** | **4 models concurrent** | **~28 GB** |

**Works for:** Cricket (T20/ODI/Test) and Soccer/Football

**Expected time to first reel:** ~25 minutes (including model download)

## Section 1 - Environment Setup

Run these cells first. Verify AMD GPU and install packages.

In [ ]:
# CELL 1 - Verify AMD GPU and system tools
# Must pass before proceeding. ROCm maps CUDA API so torch.cuda works on AMD.

import torch, subprocess, sys

print('='*55)
print('ENVIRONMENT CHECK')
print('='*55)
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU name: {name}')
    print(f'VRAM    : {vram:.1f} GB')
    if vram < 28:
        print('WARNING: Less than 28GB. Use GPU_MEM_UTIL=0.35 in config.')
    else:
        print('OK: Enough VRAM for all 4 models')
else:
    print('ERROR: No GPU. Select ROCm runtime in AMD Notebooks dropdown.')

# Check ffmpeg
r = subprocess.run(['ffmpeg','-version'], capture_output=True, text=True)
print(f'ffmpeg  : {"OK" if r.returncode==0 else "MISSING - run apt-get install ffmpeg"}')

# Show live GPU stats
subprocess.run(['rocm-smi','--showuse','--showmemuse'], check=False)


In [ ]:
# CELL 2 - Install all required packages
# Takes 3-5 minutes. AMD Notebooks already has PyTorch+ROCm.

import subprocess, sys

pkgs = [
    'librosa',       # crowd noise analysis
    'soundfile',     # audio I/O
    'transformers',  # Whisper
    'accelerate',    # HuggingFace
    'timm',          # EfficientNet
    'scipy',         # peak detection
    'scikit-learn',  # utilities
    'openai',        # vLLM client
    'datasets',      # SoccerNet-Echoes
    'yt-dlp',        # download demo video
    'pytesseract',   # scorecard OCR (cricket)
    'ultralytics',   # YOLOv8 player segmentation
    'matplotlib',    # visualisations
    'Pillow',        # image handling
]
for pkg in pkgs:
    r = subprocess.run(
        [sys.executable,'-m','pip','install',pkg,'--break-system-packages','-q'],
        capture_output=True, text=True)
    print(f'  {"OK" if r.returncode==0 else "FAIL"} {pkg}')
print('Done.')


## Section 2 - Configuration

**Edit this cell before running anything else.** All other cells read from these variables.

| Variable | Description |
|----------|-------------|
| SPORT | cricket_t20 / cricket_odi / cricket_test / soccer |
| VIDEO_PATH | Path to your match video |
| AD_IMAGE_PATH | Path to your ad PNG (use transparent background) |
| AD_STRATEGY | auto / hoarding / ground / screen |

In [ ]:
# CELL 3 - Global configuration - EDIT BEFORE RUNNING

from pathlib import Path

# Sport: cricket_t20 | cricket_odi | cricket_test | soccer
SPORT          = 'cricket_t20'

# Paths
VIDEO_PATH     = './data/demo_match.mp4'
AD_IMAGE_PATH  = './assets/ad_banner.png'
OUTPUT_DIR     = Path('./results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
Path('./assets').mkdir(exist_ok=True)
Path('./data').mkdir(exist_ok=True)

# Highlight detection parameters per sport
HIGHLIGHT_SETTINGS = {
    'soccer'       : {'min_height':0.55, 'min_distance':60,  'window_before':25, 'window_after':30},
    'cricket_t20'  : {'min_height':0.50, 'min_distance':30,  'window_before':12, 'window_after':25},
    'cricket_odi'  : {'min_height':0.58, 'min_distance':45,  'window_before':15, 'window_after':28},
    'cricket_test' : {'min_height':0.70, 'min_distance':120, 'window_before':20, 'window_after':35},
}

# Virtual Ad Insertion
AD_STRATEGY    = 'auto'   # auto tries: screen > hoarding > ground
AD_ALPHA       = 0.88     # opacity 0=invisible 1=fully opaque

# Models
WHISPER_MODEL  = 'openai/whisper-large-v3'
CAPTION_MODEL  = 'meta-llama/Llama-3.1-8B-Instruct'
VLLM_PORT      = 8000
GPU_MEM_UTIL   = 0.50     # VRAM fraction for vLLM (leave rest for other models)
MAX_HIGHLIGHTS = 10

# Scorecard region for cricket OCR (x, y, width, height)
# Adjust to match your broadcast overlay position
SCOREBOARD_REGION = (0, 0, 280, 55)

print(f'Config: SPORT={SPORT}, STRATEGY={AD_STRATEGY}, MAX_CLIPS={MAX_HIGHLIGHTS}')


## Section 3 - Download Data

Download the demo video and the SoccerNet-Echoes commentary dataset.

> **Replace `DEMO_VIDEO_URL`** with a Creative Commons sports video URL.
> Search YouTube for: `cricket T20 highlights Creative Commons` or `football match highlights CC`

In [ ]:
# CELL 4 - Download video from direct URL (archive.org)
import subprocess
from pathlib import Path

VIDEO_URL = "https://archive.org/download/ind_vs_pak_test_clip/ind_vs_pak_test_clip.mp4"

if not Path(VIDEO_PATH).exists():
    print(f"Downloading from archive.org...")
    result = subprocess.run([
        "wget",
        "--progress=bar:force",
        "--output-document", VIDEO_PATH,
        VIDEO_URL
    ], capture_output=False)   # capture_output=False shows live progress bar
    print("Done" if result.returncode == 0 else f"Failed: {result.returncode}")
else:
    print(f"Already exists: {VIDEO_PATH}")

# Verify
probe = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json",
     "-show_format", VIDEO_PATH],
    capture_output=True, text=True
)
if probe.returncode == 0:
    import json
    info = json.loads(probe.stdout)
    dur  = float(info["format"]["duration"])
    mb   = float(info["format"]["size"]) / 1e6
    print(f"Duration : {int(dur//60)}:{int(dur%60):02d}")
    print(f"Size     : {mb:.1f} MB")
    print(f"Path     : {VIDEO_PATH}")



In [ ]:
# CELL 5 - Download SoccerNet-Echoes and create placeholder ad

from datasets import load_dataset

print('Downloading SoccerNet-Echoes commentary dataset...')
echoes = load_dataset('SoccerNet/SN-echoes', 'whisper_v3', split='en')
print(f'Loaded {len(echoes):,} commentary segments')

# Create placeholder ad if none exists
if not Path(AD_IMAGE_PATH).exists():
    try:
        from PIL import Image, ImageDraw
        img  = Image.new('RGBA', (1280, 360), (255,255,255,0))
        draw = ImageDraw.Draw(img)
        draw.rectangle([10,10,1270,350], fill=(15,82,186,220), outline=(255,255,255,255), width=3)
        draw.text((640,180), 'YOUR AD HERE', fill='white', anchor='mm')
        img.save(AD_IMAGE_PATH)
        print(f'Placeholder ad created: {AD_IMAGE_PATH}')
        print('Replace with your actual ad image (PNG with transparency)')
    except Exception as e:
        print(f'Create ad manually at {AD_IMAGE_PATH}: {e}')


## Section 4 - Helper Functions

Run all cells in this section to define the pipeline functions. No models are loaded here.
Functions are grouped by pipeline stage.

In [ ]:
# CELL 6 - All imports

import cv2
import numpy as np
import torch
import librosa
import subprocess
import json
import time
import re
import os
from pathlib import Path
from PIL import Image
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from torchvision import transforms
import timm
from openai import OpenAI

print('All imports OK')


In [ ]:
# CELL 7 - Stream A: Crowd noise energy
# Detects excitement from crowd roar. Achieves ~89% accuracy alone.
# Returns per-second RMS energy in speech/crowd band (200-4000Hz).

def extract_crowd_energy(video_path, window_sec=1):
    """Extract normalised crowd excitement per second from video audio."""
    y, sr = librosa.load(video_path, sr=22050, mono=True)
    y_f   = librosa.effects.preemphasis(y, coef=0.95)
    rms   = librosa.feature.rms(y=y_f, frame_length=sr, hop_length=sr*window_sec)
    e     = rms[0]
    e_n   = (e - e.min()) / (e.max() - e.min() + 1e-9)
    return np.convolve(e_n, np.ones(5)/5, mode='same')

print('extract_crowd_energy() defined')


In [ ]:
# CELL 8 - Stream B: Commentary keywords and scoring
# Whisper transcribes audio; keywords map excitement to score 0-1.

EXCITEMENT_KEYWORDS = {
    'soccer': {
        1.0: ['GOAL','what a goal','scored','into the net'],
        0.9: ['incredible','unbelievable','magnificent','what a save'],
        0.8: ['penalty','red card','free kick'],
        0.3: ['offside','foul','yellow card'],
    },
    'cricket_t20': {
        1.0: ['SIX','six!','SIXER','chhakka','OUT','WICKET','CAUGHT',
              'caught!','BOWLED','bowled','LBW','run out','stumped'],
        0.9: ['incredible','unbelievable','century','hundred','hat trick'],
        0.8: ['FOUR','four!','chauka','boundary','fifty','maiden'],
        0.6: ['DRS','review','no ball','free hit'],
        0.3: ['wide','dot ball','single'],
    },
}
EXCITEMENT_KEYWORDS['cricket_odi']  = EXCITEMENT_KEYWORDS['cricket_t20']
EXCITEMENT_KEYWORDS['cricket_test'] = EXCITEMENT_KEYWORDS['cricket_t20']

def score_commentary(text, sport='cricket_t20'):
    """Return excitement score 0-1 for a commentary text segment."""
    kws = EXCITEMENT_KEYWORDS.get(sport, EXCITEMENT_KEYWORDS['cricket_t20'])
    score, t = 0.0, text.lower()
    for weight, kw_list in kws.items():
        if any(k.lower() in t for k in kw_list):
            score = max(score, weight)
    return score

def transcribe_and_score(video_path, sport='cricket_t20'):
    """Transcribe video with Whisper and score each segment. Uses whisper_pipe global."""
    result = whisper_pipe(video_path, chunk_length_s=30, stride_length_s=5)
    scored = []
    for chunk in result.get('chunks', []):
        s, e = chunk['timestamp']
        text = chunk.get('text', '')
        scored.append({'start': float(s or 0), 'end': float(e or 0),
                       'text': text, 'score': score_commentary(text, sport)})
    return scored

print('Commentary helpers defined')


In [ ]:
# CELL 9 - Stream C: Video frame scorer
# EfficientNet-B0 classifies frames into excitement categories.
# Pre-trained ImageNet weights work as a baseline. Fine-tune for better results.

VISUAL_CLASSES = {
    'soccer': {
        'labels' : ['CELEBRATION','ACTION_SHOT','CLOSE_UP','CROWD','REPLAY','ROUTINE'],
        'weights': [1.0, 0.8, 0.5, 0.4, 0.2, 0.0],
    },
    'cricket_t20': {
        'labels' : ['SIX_HIT','BOUNDARY','WICKET','UMPIRE_SIGNAL',
                    'CELEBRATION','CROWD','ROUTINE'],
        'weights': [1.0, 0.85, 1.0, 0.95, 0.8, 0.4, 0.0],
    },
}
VISUAL_CLASSES['cricket_odi']  = VISUAL_CLASSES['cricket_t20']
VISUAL_CLASSES['cricket_test'] = VISUAL_CLASSES['cricket_t20']

IMG_TRANSFORM = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def score_frames(video_path, sport='cricket_t20', sample_fps=1):
    """Sample frames and score visual excitement. Uses effi_model global."""
    weights = VISUAL_CLASSES.get(sport, VISUAL_CLASSES['cricket_t20'])['weights']
    cap     = cv2.VideoCapture(video_path)
    fps     = cap.get(cv2.CAP_PROP_FPS)
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step    = max(1, int(fps/sample_fps))
    scores, batch = [], []
    for i in range(0, total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret: break
        img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        batch.append(IMG_TRANSFORM(img))
        if len(batch) == 32:
            with torch.no_grad():
                probs = torch.softmax(effi_model(torch.stack(batch).cuda()), dim=1).cpu().numpy()
                for p in probs:
                    scores.append(float(sum(px*w for px,w in zip(p,weights))))
            batch = []
    if batch:
        with torch.no_grad():
            probs = torch.softmax(effi_model(torch.stack(batch).cuda()), dim=1).cpu().numpy()
            for p in probs:
                scores.append(float(sum(px*w for px,w in zip(p,weights))))
    cap.release()
    return np.array(scores)

print('score_frames() defined')


In [ ]:
# CELL 10 - Fusion and peak detection
# Combines all 3 signals into one timeline and finds highlight moments.

def fuse_modalities(crowd, commentary, video, duration, weights=(0.4,0.3,0.3)):
    """Weighted fusion of 3 excitement signals into one timeline."""
    T   = duration
    tc  = np.interp(np.arange(T), np.linspace(0,T,len(crowd)), crowd)
    tv  = np.interp(np.arange(T), np.linspace(0,T,len(video)), video)
    tco = np.zeros(T)
    for seg in commentary:
        s,e = max(0,int(seg['start'])), min(T, int(seg['end'])+1)
        tco[s:e] = max(tco[s:e].max(), seg['score'])
    fused = weights[0]*tc + weights[1]*tco + weights[2]*tv
    return gaussian_filter1d(fused, sigma=3)

def detect_highlights(excitement, sport='cricket_t20', duration=None, max_n=10):
    """Find highlight moments from fused excitement timeline."""
    p = HIGHLIGHT_SETTINGS[sport]
    peaks, _ = find_peaks(excitement, height=p['min_height'],
                          distance=p['min_distance'], prominence=0.10)
    D = duration or len(excitement)
    highlights = []
    for peak in peaks[:max_n]:
        highlights.append({
            'peak_sec'   : int(peak),
            'clip_start' : max(0, peak - p['window_before']),
            'clip_end'   : min(D,  peak + p['window_after']),
            'peak_score' : round(float(excitement[peak]), 4),
            'commentary' : '',
        })
    print(f'Found {len(highlights)} highlights')
    for h in highlights:
        t = h['peak_sec']
        print(f'  {t//60:02d}:{t%60:02d}  score={h["peak_score"]:.3f}')
    return highlights

print('fuse_modalities() and detect_highlights() defined')


In [ ]:
# CELL 11 - ffmpeg helpers
# Extract clips, add caption overlays, concatenate into reel.

def extract_clip(video_path, start_sec, end_sec, output_path):
    r = subprocess.run([
        'ffmpeg','-y','-ss',str(start_sec),'-to',str(end_sec),
        '-i',video_path,'-c:v','libx264','-crf','22',
        '-c:a','aac','-loglevel','quiet',output_path
    ], capture_output=True)
    return r.returncode == 0

def add_caption_overlay(clip_path, title, score, output_path):
    safe  = re.sub(r"[':,]", ' ', title)[:60]
    badge = f'Score: {score:.2f}'
    vf = (
        f"drawtext=text='{safe}':fontsize=22:fontcolor=white:"
        f"x=(w-text_w)/2:y=h-55:box=1:boxcolor=black@0.6:boxborderw=5,"
        f"drawtext=text='{badge}':fontsize=16:fontcolor=yellow:"
        f"x=10:y=10:box=1:boxcolor=black@0.5:boxborderw=3"
    )
    r = subprocess.run([
        'ffmpeg','-y','-i',clip_path,'-vf',vf,
        '-c:a','copy','-loglevel','quiet',output_path
    ], capture_output=True)
    return r.returncode == 0

def concatenate_clips(clip_paths, output_path):
    lst = str(OUTPUT_DIR / '_concat.txt')
    with open(lst,'w') as f:
        for cp in clip_paths:
            f.write(f"file '{os.path.abspath(cp)}'\n")
    r = subprocess.run([
        'ffmpeg','-y','-f','concat','-safe','0',
        '-i',lst,'-c','copy','-loglevel','quiet',output_path
    ], capture_output=True)
    return r.returncode == 0

print('ffmpeg helpers defined')


In [ ]:
# CELL 12 - VAI: Surface detection
# Find grass, hoardings, and sight screens in video frames.
# Returns candidates sorted by suitability score.

def _bbox_to_pts(x, y, w, h, tilt=0.0):
    inset = int(w * tilt)
    return np.float32([[x+inset,y],[x+w-inset,y],[x+w,y+h],[x,y+h]])

def _order_pts(pts):
    rect = np.zeros((4,2), dtype=np.float32)
    s, diff = pts.sum(axis=1), np.diff(pts, axis=1)
    rect[0]=pts[np.argmin(s)]; rect[2]=pts[np.argmax(s)]
    rect[1]=pts[np.argmin(diff)]; rect[3]=pts[np.argmax(diff)]
    return rect

def detect_surfaces(frame):
    """Find candidate surfaces for ad placement. Returns sorted list."""
    h, w  = frame.shape[:2]
    hsv   = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cands = []

    # Ground (grass)
    gm = cv2.inRange(hsv, np.array([32,40,40]), np.array([90,255,220]))
    k  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(20,20))
    gm = cv2.morphologyEx(cv2.morphologyEx(gm,cv2.MORPH_OPEN,k), cv2.MORPH_CLOSE,k)
    for cnt in cv2.findContours(gm,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)[0]:
        area = cv2.contourArea(cnt)
        if area < 15000: continue
        x,y,bw,bh = cv2.boundingRect(cnt)
        patch = gray[y:y+bh, x:x+bw]
        ed    = cv2.Canny(patch,50,150).mean()
        score = (area/50000)*0.4 + (1-patch.std()/128)*0.35 + (y/h)*0.25 - ed*0.01
        cands.append({'type':'ground','bbox':(x,y,bw,bh),'score':float(score),
                      'corner_pts':_bbox_to_pts(x,y,bw,bh,tilt=0.15)})

    # Hoardings
    edges = cv2.Canny(gray,50,150)
    for cnt in cv2.findContours(edges,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)[0]:
        area = cv2.contourArea(cnt)
        if not (8000 < area < w*h*0.25): continue
        approx = cv2.approxPolyDP(cnt, 0.04*cv2.arcLength(cnt,True), True)
        if len(approx) not in [4,5]: continue
        x,y,bw,bh = cv2.boundingRect(cnt)
        if not (1.5 < bw/max(bh,1) < 12): continue
        ed    = cv2.Canny(gray[y:y+bh,x:x+bw],50,150).mean()
        score = (bw/w)*0.5 + (1-ed/100)*0.3 + (y/h)*0.2
        pts   = approx.reshape(-1,2).astype(np.float32)
        cp    = _order_pts(pts) if len(pts)==4 else _bbox_to_pts(x,y,bw,bh)
        cands.append({'type':'hoarding','bbox':(x,y,bw,bh),'score':float(score),
                      'corner_pts':cp})

    # Sight screen (cricket)
    wm = cv2.inRange(hsv, np.array([0,0,200]), np.array([180,30,255]))
    k2 = np.ones((25,25),np.uint8)
    wm = cv2.morphologyEx(cv2.morphologyEx(wm,cv2.MORPH_OPEN,k2), cv2.MORPH_CLOSE,k2)
    for cnt in cv2.findContours(wm,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)[0]:
        area = cv2.contourArea(cnt)
        if area < 20000: continue
        x,y,bw,bh = cv2.boundingRect(cnt)
        score = (area/40000)*0.7 + (bw/w)*0.3 + 0.3
        cands.append({'type':'screen','bbox':(x,y,bw,bh),'score':float(score),
                      'corner_pts':_bbox_to_pts(x,y,bw,bh)})

    return sorted(cands, key=lambda c: c['score'], reverse=True)

print('detect_surfaces() defined')


In [ ]:
# CELL 13 - VAI: Perspective warp and alpha blend
# Warps ad to match surface orientation and blends it into the frame.
# Brightness matching makes the ad look natural in different lighting.

def warp_ad_to_surface(frame, ad_img, dst_pts, alpha=0.88, brightness_match=True):
    """Warp ad_img onto surface defined by dst_pts (4 corners). Returns (frame, M)."""
    h, w   = frame.shape[:2]
    ah, aw = ad_img.shape[:2]
    src    = np.float32([[0,0],[aw,0],[aw,ah],[0,ah]])
    M      = cv2.getPerspectiveTransform(src, dst_pts)
    warped = cv2.warpPerspective(ad_img[:,:,:3], M, (w,h),
                                  flags=cv2.INTER_LINEAR, borderValue=0)
    mask   = np.zeros((h,w), dtype=np.float32)
    cv2.fillConvexPoly(mask, dst_pts.astype(np.int32), 1.0)
    if ad_img.shape[2] == 4:
        am   = cv2.warpPerspective(ad_img[:,:,3].astype(np.float32)/255, M, (w,h))
        mask = mask * am
    if brightness_match:
        sp = frame[mask>0.3]; wp = warped[mask>0.3]
        if len(sp)>100 and len(wp)>100:
            sb = float(cv2.cvtColor(sp.reshape(1,-1,3),cv2.COLOR_BGR2GRAY).mean())
            wb = float(cv2.cvtColor(wp.reshape(1,-1,3),cv2.COLOR_BGR2GRAY).mean())
            if wb > 1:
                warped = np.clip(warped.astype(np.float32)*np.clip(sb/wb,0.5,1.5),0,255).astype(np.uint8)
    m3 = np.stack([mask*alpha]*3, axis=2)
    r  = frame.astype(np.float32)*(1-m3) + warped.astype(np.float32)*m3
    return r.astype(np.uint8), M

def apply_player_occlusion(composited, original, player_mask):
    """Restore original pixels where players stand - ad appears BEHIND players."""
    if player_mask is None: return composited
    pd = cv2.dilate(player_mask, np.ones((5,5),np.uint8), iterations=2)
    m3 = np.stack([pd.astype(np.float32)/255]*3, axis=2)
    return (composited.astype(np.float32)*(1-m3) + original.astype(np.float32)*m3).astype(np.uint8)

print('VAI warp and occlusion helpers defined')


In [ ]:
# CELL 14 - VAI: Optical flow tracker (temporal consistency)
# Tracks the 4 ad corner points across frames using Lucas-Kanade optical flow.
# Without this, the ad would jump around - this keeps it locked to the surface.

class AdTracker:
    """Track 4 ad placement corners across video frames."""

    def __init__(self, anchor_frame, initial_pts):
        self.prev_gray = cv2.cvtColor(anchor_frame, cv2.COLOR_BGR2GRAY)
        self.pts = initial_pts.reshape(-1,1,2).astype(np.float32)
        self.lk  = dict(winSize=(21,21), maxLevel=3,
                        criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,30,0.01))

    def update(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        fwd, fok, _ = cv2.calcOpticalFlowPyrLK(self.prev_gray, gray, self.pts, None, **self.lk)
        bwd, bok, _ = cv2.calcOpticalFlowPyrLK(gray, self.prev_gray, fwd, None, **self.lk)
        fb_err = np.sqrt(((self.pts - bwd)**2).sum(axis=2))
        good   = (fok.ravel()==1) & (fb_err.ravel()<3.0)
        if good.sum() >= 3:
            if good.sum() == 4:
                self.pts = fwd
            else:
                H, _ = cv2.findHomography(self.pts[good], fwd[good], cv2.RANSAC, 3.0)
                if H is not None: self.pts = cv2.perspectiveTransform(self.pts, H)
        self.prev_gray = gray.copy()
        return self.pts.reshape(4,2)

print('AdTracker class defined')


In [ ]:
# CELL 15 - VAI: Per-clip ad insertion pipeline
# Detects surface once per clip, tracks it across frames, inserts ad with occlusion.

def insert_ad_in_clip(clip_path, ad_image_path, output_path,
                       strategy='auto', use_occlusion=True, alpha=0.88):
    """Insert virtual ad into one highlight clip. Returns metadata dict."""
    t0     = time.time()
    cap    = cv2.VideoCapture(clip_path)
    fps, fw = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh, nf  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out    = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (fw,fh))

    ad_img = cv2.imread(ad_image_path, cv2.IMREAD_UNCHANGED)
    if ad_img is None:
        cap.release(); out.release()
        return {'status':'error','reason':f'Cannot read {ad_image_path}'}
    if ad_img.shape[2] == 3:
        ad_img = cv2.cvtColor(ad_img, cv2.COLOR_BGR2BGRA)
        ad_img[:,:,3] = int(alpha*255)

    # Detect surface from middle frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, nf//2)
    _, anchor = cap.read()
    cands = detect_surfaces(anchor)
    order = ['screen','hoarding','ground'] if strategy=='auto' else [strategy]
    best  = next((c for pref in order for c in cands if c['type']==pref), None)

    if best is None:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        while True:
            ret, f = cap.read()
            if not ret: break
            out.write(f)
        cap.release(); out.release()
        return {'status':'no_surface','output':output_path}

    print(f'    Surface: {best["type"]}  score={best["score"]:.3f}')
    tracker = AdTracker(anchor, best['corner_pts'])

    # Optional player model
    player_model = None
    if use_occlusion:
        try:
            from ultralytics import YOLO
            player_model = YOLO('yolov8n-seg.pt').to('cuda')
        except: pass

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    done = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        pts = tracker.update(frame)
        comp, _ = warp_ad_to_surface(frame, ad_img, pts, alpha=alpha)
        if player_model:
            res = player_model(frame, classes=[0], verbose=False)[0]
            pmask = None
            if res.masks:
                raw   = res.masks.data[0].cpu().numpy()
                pmask = cv2.resize((raw*255).astype(np.uint8),(fw,fh))
            comp = apply_player_occlusion(comp, frame, pmask)
        out.write(comp)
        done += 1
    cap.release(); out.release()
    return {'status':'success','placement_type':best['type'],
            'surface_score':round(best['score'],3),
            'frames_processed':done, 'latency_sec':round(time.time()-t0,1)}

print('insert_ad_in_clip() defined')


## Section 5 - Load Models

**Important:** Start vLLM first (Cell 16) - it takes ~60 seconds to initialise. Load Whisper and EfficientNet while vLLM starts in the background.

In [ ]:
# CELL 16 - Start vLLM server (Llama 3.1 8B) in background
# Run this cell FIRST. Takes ~60s to start.

import subprocess, time

vllm_proc = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model',                  CAPTION_MODEL,
    '--host',                   '0.0.0.0',
    '--port',                   str(VLLM_PORT),
    '--dtype',                  'float16',
    '--gpu-memory-utilization', str(GPU_MEM_UTIL),
    '--max-model-len',          '2048',
], stdout=open('/tmp/vllm.log','w'), stderr=subprocess.STDOUT)

print(f'vLLM started (PID {vllm_proc.pid})')
print('Load Whisper and EfficientNet while vLLM initialises...')


In [ ]:
# CELL 17 - Load Whisper large-v3 on AMD GPU
# VRAM: ~10 GB in float16. Takes ~60s to download on first run.

from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

print(f'Loading {WHISPER_MODEL}...')
t0 = time.time()
w_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL, torch_dtype=torch.float16, low_cpu_mem_usage=True).to('cuda')
w_proc  = AutoProcessor.from_pretrained(WHISPER_MODEL)
whisper_pipe = pipeline(
    'automatic-speech-recognition',
    model=w_model, tokenizer=w_proc.tokenizer,
    feature_extractor=w_proc.feature_extractor,
    return_timestamps=True, torch_dtype=torch.float16, device='cuda',
)
print(f'Whisper loaded in {time.time()-t0:.1f}s  |  VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB')


In [ ]:
# CELL 18 - Load EfficientNet-B0 for frame scoring
# VRAM: ~1 GB. Pre-trained ImageNet weights used as baseline.

n_cls = len(VISUAL_CLASSES.get(SPORT, VISUAL_CLASSES['cricket_t20'])['labels'])
effi_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=n_cls)
effi_model = effi_model.cuda().eval()
print(f'EfficientNet loaded  |  VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB  |  Classes: {n_cls}')


In [ ]:
# CELL 19 - Wait for vLLM and create client

import urllib.request

print('Waiting for vLLM...')
for i in range(120):
    try:
        urllib.request.urlopen(f'http://localhost:{VLLM_PORT}/health')
        print(f'vLLM ready (waited {i}s)')
        break
    except:
        if i % 20 == 0 and i > 0: print(f'  Still waiting... {i}s')
        time.sleep(1)
else:
    print('vLLM timeout. Check /tmp/vllm.log')
    with open('/tmp/vllm.log') as f: print(''.join(f.readlines()[-15:]))

llm_client = OpenAI(base_url=f'http://localhost:{VLLM_PORT}/v1', api_key='unused')
print(f'Total VRAM in use: {torch.cuda.memory_allocated()/1e9:.1f}GB')


In [ ]:
# CELL 20 - Caption generation function (LLM)

def generate_caption(commentary_text, timestamp, peak_score, sport='cricket_t20'):
    """Generate highlight title and social caption using Llama 3.1 8B."""
    ctx = {'cricket_t20':'T20 cricket','cricket_odi':'ODI cricket',
           'cricket_test':'Test cricket','soccer':'football'}.get(sport,'sports')
    prompt = (
        f'You are a sports broadcaster. Write content for social media.\n'
        f'Sport: {ctx}, Timestamp: {timestamp}, Excitement: {peak_score}/1.0\n'
        f'Commentary: "{commentary_text[:300]}"\n'
        f'Return ONLY JSON: {{"title": "6-8 word title", "caption": "1 sentence with 2 hashtags"}}'
    )
    try:
        resp = llm_client.chat.completions.create(
            model=CAPTION_MODEL, messages=[{'role':'user','content':prompt}],
            max_tokens=150, temperature=0.7)
        raw = re.sub(r'```json|```','', resp.choices[0].message.content).strip()
        return json.loads(raw)
    except:
        return {'title':f'Highlight at {timestamp}', 'caption':f'Great moment! #{sport}'}

print('generate_caption() defined')


## Section 6 - Full End-to-End Pipeline

This section defines and runs the master pipeline function. Edit CELL 22 to run with your video and configuration from Section 2.

In [ ]:
# CELL 21 - Master pipeline function
# Runs all stages: audio + video analysis -> fusion -> clip extraction -> ad insertion -> reel

def generate_highlight_reel(video_path, ad_image_path, sport='cricket_t20',
                              ad_strategy='auto', max_highlights=10, output_dir='./results'):
    """Full pipeline: sports video -> monetised highlight reel with AI captions."""
    t_start = time.time()
    OUT = Path(output_dir)
    OUT.mkdir(parents=True, exist_ok=True)

    # Get video duration
    probe = subprocess.run(['ffprobe','-v','quiet','-print_format','json',
                            '-show_format',video_path], capture_output=True,text=True)
    duration = int(float(json.loads(probe.stdout)['format']['duration']))
    print(f'Video: {Path(video_path).name}  Duration: {duration//60}:{duration%60:02d}  Sport: {sport}')

    # 3 audio/video streams
    print('\n[1/5] Crowd energy...')
    crowd_energy = extract_crowd_energy(video_path)
    print(f'      OK ({len(crowd_energy)}s)')

    print('[2/5] Whisper transcription...')
    commentary = transcribe_and_score(video_path, sport)
    print(f'      OK ({len(commentary)} segments)')

    print('[3/5] Frame scoring...')
    video_scores = score_frames(video_path, sport, sample_fps=1)
    print(f'      OK ({len(video_scores)} frames)')

    print('[4/5] Fusion + peak detection...')
    excitement = fuse_modalities(crowd_energy, commentary, video_scores, duration)
    highlights = detect_highlights(excitement, sport, duration, max_highlights)
    for h in highlights:
        relevant = [c['text'] for c in commentary
                    if h['clip_start'] <= c['start'] <= h['clip_end']]
        h['commentary'] = ' '.join(relevant)[:300]

    print(f'[5/5] Building {len(highlights)} clips...')
    final_clips, clip_meta = [], []
    for i, h in enumerate(highlights):
        ts = f"{h['peak_sec']//60:02d}:{h['peak_sec']%60:02d}"
        print(f'\n  Clip {i+1}/{len(highlights)} [{ts}] score={h["peak_score"]:.3f}')
        raw = str(OUT/f'clip_{i:02d}_raw.mp4')
        adp = str(OUT/f'clip_{i:02d}_ad.mp4')
        cap = str(OUT/f'clip_{i:02d}_final.mp4')
        if not extract_clip(video_path, h['clip_start'], h['clip_end'], raw): continue
        vai = insert_ad_in_clip(raw, ad_image_path, adp, strategy=ad_strategy)
        print(f'    Ad: {vai.get("status")} type={vai.get("placement_type","—")} {vai.get("latency_sec",0):.1f}s')
        cap_text = generate_caption(h['commentary'], ts, h['peak_score'], sport)
        print(f'    Caption: "{cap_text["title"]}"')
        add_caption_overlay(adp, cap_text['title'], h['peak_score'], cap)
        final_clips.append(cap)
        clip_meta.append({**h, **vai, **cap_text, 'clip_path':cap})

    reel = str(OUT/'highlight_reel.mp4')
    ok   = concatenate_clips(final_clips, reel)
    total = round(time.time()-t_start,1)
    result = {'reel_path':reel if ok else None,'sport':sport,'duration_sec':duration,
              'highlights_found':len(highlights),'clips_produced':len(final_clips),
              'total_latency_sec':total,'clips':clip_meta}
    with open(str(OUT/'metadata.json'),'w') as f: json.dump(result,f,indent=2,default=str)
    print(f'\nDONE: {total}s  |  {len(final_clips)} clips  |  Reel: {reel}')
    return result

print('generate_highlight_reel() defined')


In [ ]:
# CELL 22 - RUN THE PIPELINE
# This runs everything. Expected time: 15-30 min for 90-min match.

result = generate_highlight_reel(
    video_path     = VIDEO_PATH,
    ad_image_path  = AD_IMAGE_PATH,
    sport          = SPORT,
    ad_strategy    = AD_STRATEGY,
    max_highlights = MAX_HIGHLIGHTS,
    output_dir     = str(OUTPUT_DIR),
)

print(f'Reel: {result["reel_path"]}')
print(f'Clips: {result["clips_produced"]}  |  Latency: {result["total_latency_sec"]}s')


## Section 7 - Cricket Bonus: Scorecard OCR

Cricket only. Reads the scoreboard overlay via OCR for near-perfect event detection.

**Adjust `SCOREBOARD_REGION`** in Cell 3 to match your broadcast.
Common positions: `(0, 0, 280, 55)` for Sky Sports/Hotstar, `(0, 0, 320, 60)` for Star Sports.

In [ ]:
# CELL 23 - Scorecard OCR for cricket
# Near-perfect signal: runs +4 = FOUR, +6 = SIX, wickets +1 = WICKET

import pytesseract, re

def extract_scoreboard_events(video_path, region=None, sample_every_n_sec=2):
    """Scan broadcast scoreboard to detect cricket events with near-100% precision."""
    if region is None: region = SCOREBOARD_REGION
    rx,ry,rw,rh = region
    cap    = cv2.VideoCapture(video_path)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    events, prev = [], {'runs':-1,'wickets':-1}
    for sec in range(0, int(total//fps), sample_every_n_sec):
        cap.set(cv2.CAP_PROP_POS_FRAMES, sec*fps)
        ret, frame = cap.read()
        if not ret: break
        crop  = frame[ry:ry+rh, rx:rx+rw]
        gray  = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        _, bw = cv2.threshold(gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        text  = pytesseract.image_to_string(bw, config='--psm 7')
        m     = re.search(r'(\d{1,3})[/\-](\d{1})', text)
        if not m: continue
        runs, wkts = int(m.group(1)), int(m.group(2))
        if prev['runs'] >= 0:
            dr, dw = runs-prev['runs'], wkts-prev['wickets']
            if   dr == 6: events.append({'timestamp_sec':sec,'event':'SIX',   'score':f'{runs}/{wkts}'})
            elif dr == 4: events.append({'timestamp_sec':sec,'event':'FOUR',  'score':f'{runs}/{wkts}'})
            elif dw >= 1: events.append({'timestamp_sec':sec,'event':'WICKET','score':f'{runs}/{wkts}'})
        prev = {'runs':runs,'wickets':wkts}
    cap.release()
    print(f'OCR: {len(events)} events detected')
    return events

def boost_excitement_with_ocr(excitement, ocr_events, boost=0.40):
    """Boost excitement score at OCR-detected event timestamps."""
    boosted = excitement.copy()
    for evt in ocr_events:
        ts = evt['timestamp_sec']
        s, e = max(0,ts-3), min(len(boosted),ts+4)
        boosted[s:e] = np.minimum(1.0, boosted[s:e]+boost)
    return boosted

print('Scorecard OCR functions defined')


In [ ]:
# CELL 24 - Run cricket pipeline with OCR enhancement
# Run this cell INSTEAD of Cell 22 for cricket matches.

if SPORT.startswith('cricket'):
    print('Running enhanced cricket pipeline with scorecard OCR...')

    # Compute all signals
    crowd_energy = extract_crowd_energy(VIDEO_PATH)
    commentary   = transcribe_and_score(VIDEO_PATH, SPORT)
    video_scores = score_frames(VIDEO_PATH, SPORT, sample_fps=1)

    probe    = subprocess.run(['ffprobe','-v','quiet','-print_format','json',
                               '-show_format',VIDEO_PATH], capture_output=True,text=True)
    duration = int(float(json.loads(probe.stdout)['format']['duration']))

    excitement = fuse_modalities(crowd_energy, commentary, video_scores, duration)

    # Boost with OCR
    ocr_events = extract_scoreboard_events(VIDEO_PATH)
    print(f'OCR: {sum(1 for e in ocr_events if e["event"]=="SIX")} sixes, '
          f'{sum(1 for e in ocr_events if e["event"]=="FOUR")} fours, '
          f'{sum(1 for e in ocr_events if e["event"]=="WICKET")} wickets')
    excitement_boosted = boost_excitement_with_ocr(excitement, ocr_events)

    highlights = detect_highlights(excitement_boosted, SPORT, duration, MAX_HIGHLIGHTS)
    print(f'Enhanced detection: {len(highlights)} highlights')
else:
    print('Not a cricket match. Use Cell 22 instead.')


## Section 8 - Metrics for Slide 4

Capture GPU stats, latency, and throughput for the hackathon presentation.

In [ ]:
# CELL 25 - GPU metrics for Slide 4

import torch

print('='*55)
print('GPU METRICS - FOR HACKATHON SLIDE 4')
print('='*55)

alloc = torch.cuda.memory_allocated()/1e9
total = torch.cuda.get_device_properties(0).total_memory/1e9
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM allocated: {alloc:.1f} GB / {total:.1f} GB ({100*alloc/total:.0f}%)')
print(f'Models loaded: Whisper(~10GB) + EfficientNet(~1GB) + Llama3.1-8B(~16GB)')

if 'result' in dir() and result:
    print(f'\nPipeline performance:')
    print(f'  Video duration  : {result.get("duration_sec","?")//60}min')
    print(f'  Total latency   : {result.get("total_latency_sec","?")}s')
    print(f'  Highlights found: {result.get("highlights_found","?")}')
    print(f'  Clips with ad   : {result.get("clips_produced","?")}')

subprocess.run(['rocm-smi','--showuse','--showmemuse','--showpower'], check=False)


In [ ]:
# CELL 26 - Excitement timeline visualisation

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if 'excitement' not in dir():
    print('Run the pipeline first, then re-run this cell.')
else:
    fig, axes = plt.subplots(4,1,figsize=(16,10),sharex=True)
    T = np.arange(len(excitement))
    tc = np.interp(T, np.linspace(0,len(T),len(crowd_energy)), crowd_energy)
    tv = np.interp(T, np.linspace(0,len(T),len(video_scores)), video_scores)
    tco = np.zeros(len(T))
    for seg in commentary:
        s,e = max(0,int(seg['start'])), min(len(T),int(seg['end'])+1)
        tco[s:e] = max(tco[s:e].max(), seg['score'])

    for ax,(sig,c,lbl) in zip(axes[:3],[
        (tc,'#534AB7','Crowd energy'),(tco,'#1D9E75','Commentary'),(tv,'#BA7517','Visual')]):
        ax.fill_between(T,sig,alpha=0.4,color=c); ax.plot(T,sig,color=c,lw=0.8)
        ax.set_ylabel(lbl,fontsize=10)

    axes[3].fill_between(T,excitement,alpha=0.3,color='#D85A30')
    axes[3].plot(T,excitement,color='#D85A30',lw=1.5)
    if 'highlights' in dir():
        for h in highlights:
            for ax in axes: ax.axvline(h['peak_sec'],color='red',alpha=0.4,lw=1,ls='--')
            axes[3].scatter(h['peak_sec'],excitement[h['peak_sec']],s=80,color='red',zorder=5)

    def fmt_t(x,_): return f'{int(x)//60}:{int(x)%60:02d}'
    axes[3].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_t))
    axes[3].set_xlabel('Time',fontsize=10); axes[3].set_ylabel('Fused',fontsize=10)
    for ax in axes: ax.set_ylim(0,1.05); ax.grid(axis='x',alpha=0.2)
    plt.suptitle(f'Excitement Timeline - {SPORT}', fontsize=13)
    plt.tight_layout()
    out_p = str(OUTPUT_DIR/'timeline.png')
    plt.savefig(out_p,dpi=150,bbox_inches='tight')
    print(f'Saved: {out_p}')
    plt.show()
